# Modeling

### Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import umap

from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

### Constants

In [ ]:
PROCESSED_TRSHMNT_DIR_FROM_CWD = "../data/processed/transshipment"
PROCESSED_LOITERING_FILENAME = "processed-loitering-events.csv"

RANDOM_SEED = 21

K_MEANS_N_INIT = 10

### Set Seed for Reproducible Random Number Generation

In [ ]:
np.random.seed(RANDOM_SEED)

## Modeling Loitering Events

### Load Processed Dataset

In [ ]:
loitering_df = pd.read_csv(
    os.path.join(
        os.getcwd(),
        PROCESSED_TRSHMNT_DIR_FROM_CWD,
        PROCESSED_LOITERING_FILENAME
    )
)
loitering_df.info()

### 1. Principal Component Analysis (PCA)

Although there are not that many features, principal component analysis is still useful for removing possible noise from the dataset, because the events in the dataset are best guesses and not objectively confirmed transshipment events. This provides significantly better clusters when using K-Means, as K-Means is sensitive to noise and outliers.

#### 1. Select and Scale Features

In [ ]:
loitering_PCA_features = ['total_event_duration', 'median_speed_knots', 'nearest_mpa_dist', 'movement_mag', 'local_hour', 'is_night_corrected']

loitering_PCA_X = loitering_df[loitering_PCA_features]

loitering_PCA_scaler = StandardScaler()
loitering_PCA_X_scaled = loitering_PCA_scaler.fit_transform(loitering_PCA_X)

These features are selected because they provide a good picture of how a transshipment vessel is behaving in a possible loitering event, while not introducing a lot of noise and variance that could result in poor clusters.

#### 2. Analyze Explained Variance

In [ ]:
loitering_PCA_full = PCA()
loitering_PCA_full.fit(loitering_PCA_X_scaled)
cum_sum = np.cumsum(loitering_PCA_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_sum) + 1), cum_sum, marker='o', linestyle='--', color='b')
plt.title('Using Five Components Reduces Noise while Retaining Important Info')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid()
plt.show()

Each PC explains 10-20% varience. Using 5 components explains ~88% of the variance. Reducing to 5 components should reduce noise from possible outlier events that the dataset's methodology might've catched, while retaining enough info to create clean, descriptive clusters.

#### 3. Reduce Features to 5 Principal Components

In [ ]:
n_components = 5
loitering_PCA = PCA(n_components=n_components)
loitering_X_PCA = loitering_PCA.fit_transform(loitering_PCA_X_scaled)

In [ ]:
loadings = pd.DataFrame(
    loitering_PCA.components_.T,
    columns=[f"PC{i}" for i in range(1, n_components+1)],
    index=loitering_PCA_features
)

print(loadings)

#### Principal Component (PC) Interpretation

| Component | Dominant Features | Behavioral Label | Interpretation |
| ----------|-------------------|------------------|----------------|
 | PC1 | median_speed_knots(+0.54) total_event_duration(-0.48) nearest_mpa_dist(+0.47) | Persistence vs. Transit | Captures the trade-off between moving and staying. Low values indicate long-duration loitering at low speeds, while high values represent faster transit further from MPAs. |
| PC2 | is_night_corrected(+0.59) local_hour(+0.52) total_event_duration(-0.42) | Nocturnal Cover | Distinguishes between day and night activity. High values represent nighttime behavior, potentially used to mask activities from visual surveillance. |
| PC3 | movement_mag(+0.96) | Stationary vs. Drift | Almost exclusively captures spatial displacement. Distinguishes between vessels that are anchored or tied up (low movement) and those drifting with currents (high movement). |
| PC4 | nearest_mpa_dist(+0.74) is_night_corrected(+0.48) | Contextual Stealth | Identifies behavior relative to protected zones. High values indicate nighttime loitering specifically in the open ocean (high distance), while low values indicate proximity to MPAs. |
| PC5 | local_hour(+0.60) is_night_corrected(-0.55) median_speed_knots(+0.42) | Dusk/Dawn Transit | Captures the relationship between local time and speed. Likely distinguishes "commute" behavior or vessels moving quickly just before or after the night threshold. |

### 2. K-Means Clustering

#### 1. Test Clusters with Varying Ks

In [ ]:
wcss = []
silhouette_avg = []
k_range = range(2, 10)

for k in k_range:
    k_means = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=K_MEANS_N_INIT)
    cluster_labels = k_means.fit_predict(loitering_X_PCA)

    # WCSS (Inertia)
    wcss.append(k_means.inertia_)

    # Silhouette Score (Closer to 1 is better)
    silhouette_avg.append(silhouette_score(loitering_X_PCA, cluster_labels))

In [ ]:
# Plotting Elbow and Silhouette side-by-side
fig, ax1 = plt.subplots(figsize=(12, 5))

# Elbow Plot (WCSS)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('WCSS (Inertia)', color='tab:blue')
ax1.plot(k_range, wcss, marker='o', color='tab:blue', label='WCSS')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Silhouette Plot
ax2 = ax1.twinx()
ax2.set_ylabel('Silhouette Score', color='tab:red')
ax2.plot(k_range, silhouette_avg, marker='s', color='tab:red', label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Six Clusters Provides Decent Density and Separation')
plt.show()

Choosing a model with K=6 clusters results in the highest silhouette score and a relatively low WCSS value where WCSS starts to level off, suggest it will provide the most well-boundaried, most interpretable clusters out of the other Ks. 

#### 2. Create Model with K=6 Clusters

In [ ]:
chosen_k = 6

k_means_final = KMeans(n_clusters=chosen_k, random_state=RANDOM_SEED, n_init=K_MEANS_N_INIT)
cluster_labels = k_means_final.fit_predict(loitering_X_PCA)

centers = k_means_final.cluster_centers_
cls_labels = [f"Cluster {i}" for i in range(len(centers))]

cls_means = pd.DataFrame(centers, columns=loadings.columns, index=cls_labels)
print('Mean Principal Components for Each Cluster')
print(cls_means)

#### Cluster Interpretation

| Cluster Num | PCA Signatures |	Behavioral Label	| Description |
|-------------|----------------|------------------------|-------------|
| Cluster 0 |	Moderate PC1, Low PC2 |	Standard Daytime Operations |	Routine maritime activity occurring during the day with average speed and duration, typically located at a safe distance from protected zones. |
| Cluster 1 |	Low PC1, High PC2 |	Nighttime Stealth Loitering	Highest Risk. | These vessels exhibit long event durations at low speeds specifically during night hours. This is a primary indicator of suspicious activity near MPA borders. |
| Cluster 2 |	High PC1, Very Low PC5 | 	Nocturnal Transit |	Vessels moving at moderate speeds during the evening or night. Likely represents "commuter" behavior or ships repositioning for the next day's work. |
| Cluster 3 |	Extremely Low PC1 & PC2 |	Persistent Daytime Loitering |	These vessels stay stationary for the longest periods but operate primarily during daylight. Likely large tankers or supply vessels engaged in legitimate, long-term transshipment. |
| Cluster 4 |	High PC1, High PC4 |	Open-Sea Transit 	Low Risk. | Characterized by higher speeds, shorter durations, and significant distance from MPAs. These are likely merchant vessels passing through the region. |
| Cluster 5 | 	Massive PC3, Low PC4 |	High-Drift MPA Proximity |	Unique outliers that show extreme spatial displacement (drift) while remaining very close to MPA boundaries. Could represent vessels "grazing" protected edges due to currents. |

#### 3. Visualize Chosen Model

##### UMAP Reduction

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED)
loitering_X_UMAP = reducer.fit_transform(loitering_X_PCA)

In [ ]:
fig = px.scatter(x=loitering_X_UMAP[:, 0], y=loitering_X_UMAP[:, 1],
                 color=cluster_labels.astype(str),
                 title=f'K-Means Clusters (K={chosen_k}) - UMAP Reduction')
fig.show()

##### Clusters Illustrated on a Map

In [ ]:
# Prepare the data
loitering_nearest_mpa = loitering_df.copy()
loitering_nearest_mpa['cluster'] = cluster_labels.astype(int)
loitering_nearest_mpa['pca_1'] = loitering_X_PCA[:, 0]
loitering_nearest_mpa['pca_2'] = loitering_X_PCA[:, 1]

# Map cluster numbers to behavioral labels
cluster_names = {
    1: "Nighttime MPA Border Activity (Suspicious)",
    3: "Long-Term Daytime Loitering",
    4: "Open-Sea High-Speed Transit",
    5: "High-Drift MPA Proximity",
    0: "Standard Daytime Operations",
    2: "Nocturnal Transit/Evening Activity"
}
loitering_nearest_mpa['behavior_type'] = loitering_nearest_mpa['cluster'].map(cluster_names)

# 2. Initialize Dash App
app = Dash(__name__)

app.layout = html.Div([
    html.Div([
        html.H1("Vessel Clustering & MPA Analytics", style={'textAlign': 'center', 'fontFamily': 'Arial'}),
        html.P("Interactive Behavioral Dashboard", 
               style={'textAlign': 'center', 'color': '#7f8c8d'})
    ], style={'padding': '20px', 'backgroundColor': '#f8f9fa', 'borderBottom': '1px solid #ddd'}),

    html.Div([
        # Sidebar
        html.Div([
            html.H4("Analysis Controls"),
            html.Label("Select Behavior Cluster:"),
            dcc.Dropdown(
                id='cluster-dropdown',
                # Added 'All Clusters' option
                options=[{'label': "All Clusters", "value": "ALL"}] + 
                        [{'label': f"Cluster {k}: {v}", 'value': k} for k, v in cluster_names.items()],
                value="ALL", 
                clearable=False
            ),
            html.Hr(),
            html.Div(id='cluster-stats', style={'fontSize': '14px'})
        ], style={'width': '20%', 'display': 'inline-block', 'padding': '20px', 'verticalAlign': 'top'}),

        # Main Visualization Pane
        html.Div([
            dcc.Graph(id='global-map', style={'height': '600px'}),
        ], style={'width': '78%', 'display': 'inline-block'})
    ])
])

# 3. Callbacks for Interactivity
@app.callback(
    [Output('global-map', 'figure'),
     Output('cluster-stats', 'children')],
    [Input('cluster-dropdown', 'value')]
)
def update_dashboard(selected_cluster):
    
    # Logic to handle 'ALL' or specific cluster selection
    if selected_cluster == "ALL":
        display_df = loitering_nearest_mpa
        title_suffix = "All Behavior Profiles"
    else:
        display_df = loitering_nearest_mpa[loitering_nearest_mpa['cluster'] == selected_cluster]
        title_suffix = f"Profile: {cluster_names[selected_cluster]}"
    
    # Map Visualization
    # Using 'color' automatically generates a legend
    fig_map = px.scatter_map(
        display_df, 
        lat="starting_latitude", 
        lon="starting_longitude",
        color="behavior_type", # This creates the legend and color coding
        hover_name="transshipment_mmsi", 
        hover_data=["total_event_duration", "nearest_mpa_name", "median_speed_knots"],
        zoom=1, 
        height=600,
        title=f"Global Vessel Distribution - {title_suffix}",
        # Custom color map to highlight the suspicious cluster 1 in red
        color_discrete_map={
            cluster_names[1]: "#e74c3c", # Red
            cluster_names[3]: "#f1c40f", # Yellow
            cluster_names[4]: "#2ecc71", # Green
            cluster_names[5]: "#9b59b6", # Purple
            cluster_names[0]: "#3498db", # Blue
            cluster_names[2]: "#e67e22"  # Orange
        }
    )
    
    fig_map.update_layout(
        mapbox_style="carto-darkmatter", 
        margin={"r":0,"t":40,"l":0,"b":0},
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.5)")
    )

    # Statistical summaries
    stats = [
        html.P([html.B("Total Events: "), f"{len(display_df):,}"]),
        html.P([html.B("Avg Duration: "), f"{display_df['total_event_duration'].mean():.2f} hrs"]),
        html.P([html.B("Avg Speed: "), f"{display_df['median_speed_knots'].mean():.2f} knots"])
    ]

    return fig_map, stats

if __name__ == '__main__':
    app.run(debug=True)